In [2]:
import numpy as np
import pandas as pd
import time
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Qiskit Imports
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_algorithms.utils import algorithm_globals
from qiskit_aer import Aer

# Set seeds for reproducibility
algorithm_globals.random_seed = 42
np.random.seed(42)

class PegasosQSVM_Lazy:
    def __init__(self, num_qubits=2, lambda_reg=0.02, iterations=200):
        self.lambda_reg = lambda_reg
        self.iterations = iterations
        self.alphas = {}  # Dictionary to store non-zero alphas: {index: value}
        self.support_vectors_x = [] # List to store actual data of support vectors
        self.support_vectors_y = [] # List to store labels of support vectors
        
        # Use Aer Statevector Simulator for speed and exact results
        self.backend = Aer.get_backend('statevector_simulator')
        
        # ---------------------------------------------------------
        # CUSTOM FEATURE MAP (RY Rotations)
        # ---------------------------------------------------------
        # 1. Create a ParameterVector to represent input features
        theta = ParameterVector("theta", num_qubits)
        
        # 2. Create the Quantum Circuit
        self.feature_map = QuantumCircuit(num_qubits)
        
        # 3. Apply RY rotations for each feature
        for i in range(num_qubits):
            self.feature_map.ry(theta[i], i)
            
        # Note: We do NOT bind parameters here. The kernel binds them to X data later.
        # ---------------------------------------------------------
        
        # Kernel initialized with the custom map
        self.kernel = FidelityQuantumKernel(feature_map=self.feature_map)

    def fit(self, X, y):
        n_samples = X.shape[0]
        print(f"Training on FULL dataset ({n_samples} samples) using Lazy Pegasos...")
        print(f"Total Iterations: {self.iterations}")
        
        start_time = time.time()
        
        # Reset model
        self.alphas = {} 
        self.support_vectors_x = []
        self.support_vectors_y = []
        
        # Iterate T times
        for t in range(1, self.iterations + 1):
            # Pick one random sample i
            i = np.random.randint(0, n_samples)
            x_i = X[i]
            y_i = y[i]
            
            decision_value = 0
            
            if len(self.support_vectors_x) > 0:
                # Compute kernel between this one sample x_i and all existing support vectors
                K_values = self.kernel.evaluate(
                    x_vec=np.array([x_i]), 
                    y_vec=np.array(self.support_vectors_x)
                ).flatten()
                
                # Sum up the contributions
                for idx, k_val in enumerate(K_values):
                    decision_value += self.alphas[idx] * self.support_vectors_y[idx] * k_val

            # Learning rate decay
            eta = 1.0 / (self.lambda_reg * t)
            
            # Check Hinge Loss condition: y_i * f(x_i) < 1
            if y_i * decision_value * eta < 1:
                self.support_vectors_x.append(x_i)
                self.support_vectors_y.append(y_i)
                self.alphas[len(self.alphas)] = 1.0 
                
            if t % 20 == 0:
                print(f"   Step {t}/{self.iterations} - Support Vectors: {len(self.support_vectors_x)}")

        print(f"Training finished in {time.time() - start_time:.2f} seconds.")
        print(f"Final model has {len(self.support_vectors_x)} support vectors.")

    def predict(self, X_test):
        # We compute kernel between Test Set and Support Vectors
        if len(self.support_vectors_x) == 0:
            return np.zeros(len(X_test))
            
        K_matrix = self.kernel.evaluate(x_vec=X_test, y_vec=np.array(self.support_vectors_x))
        
        predictions = []
        for i in range(X_test.shape[0]):
            score = 0
            for j in range(len(self.support_vectors_x)):
                score += self.alphas[j] * self.support_vectors_y[j] * K_matrix[i, j]
            predictions.append(1 if score >= 0 else -1)
            
        return np.array(predictions)

def preprocess_data_full(filepath):
    print("Loading full dataset...")
    df = pd.read_csv(filepath)

    # Clean and Preprocess
    cols_to_drop = ['Debate', 'account_id', 'post_id', 'Post URL']
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    df['share_count'] = df['share_count'].fillna(0)
    df = df.dropna(subset=['reaction_count', 'comment_count'])
    df = df[df['Rating'] != 'no factual content']
    
    label_map = {'mostly true': 1, 'mostly false': -1, 'mixture of true and false': -1}
    df['Rating'] = df['Rating'].map(label_map)
    
    categorical_cols = ['Category', 'Page', 'Date Published', 'Post Type']
    le = LabelEncoder()
    for col in categorical_cols:
        if col in df.columns:
            df[col] = le.fit_transform(df[col].astype(str))
            
    y = df['Rating'].values
    X = df.drop(columns=['Rating']).values
    
    return X, y

def main():
    file_name = 'facebook-fact-check.csv'
    try:
        X_raw, y = preprocess_data_full(file_name)
    except FileNotFoundError:
        print("Dataset not found. Please ensure 'facebook-fact-check.csv' is in the directory.")
        return

    # PCA to 2 Dimensions
    print("Applying PCA...")
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_raw)
    
    # Scale to [0, 2pi] for full rotation coverage
    scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
    X_scaled = scaler.fit_transform(X_pca)

    # Split
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

    # Train Optimized Model
    model = PegasosQSVM_Lazy(num_qubits=2, iterations=200, lambda_reg=0.02)
    model.fit(X_train, y_train)

    # Evaluate
    print("\nEvaluating Training Set...")
    y_train_pred = model.predict(X_train)
    train_acc = accuracy_score(y_train, y_train_pred)

    print(f"\nEvaluating Test Set ({X_test.shape[0]} samples)...")
    y_test_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_test_pred)
    
    print("\n" + "="*40)
    print(" FINAL RESULTS ")
    print("="*40)
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Testing Accuracy:  {test_acc:.4f}")
    print("-" * 20)
    print(f"Test Precision:    {precision_score(y_test, y_test_pred, zero_division=0):.4f}")
    print(f"Test Recall:       {recall_score(y_test, y_test_pred, zero_division=0):.4f}")
    print(f"Test F1-Score:     {f1_score(y_test, y_test_pred, zero_division=0):.4f}")
    print("="*40 + "\n")

if __name__ == "__main__":
    main()

Loading full dataset...
Applying PCA...
Training on FULL dataset (1512 samples) using Lazy Pegasos...
Total Iterations: 200
   Step 20/200 - Support Vectors: 5
   Step 40/200 - Support Vectors: 9
   Step 60/200 - Support Vectors: 15
   Step 80/200 - Support Vectors: 22
   Step 100/200 - Support Vectors: 36
   Step 120/200 - Support Vectors: 42
   Step 140/200 - Support Vectors: 55
   Step 160/200 - Support Vectors: 60
   Step 180/200 - Support Vectors: 71
   Step 200/200 - Support Vectors: 78
Training finished in 3.19 seconds.
Final model has 78 support vectors.

Evaluating Training Set...

Evaluating Test Set (504 samples)...

 FINAL RESULTS 
Training Accuracy: 0.8214
Testing Accuracy:  0.8393
--------------------
Test Precision:    0.8410
Test Recall:       0.9976
Test F1-Score:     0.9126

